# PPG Time Series Exploratory Data Analysis

This notebook loads the processed PPG data, checks data quality, summarizes the signal columns, recreates the time-series plots, and saves updated report CSVs under `artifacts/reports`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "artifacts" / "processed_data"
REPORT_DIR = PROJECT_ROOT / "artifacts" / "reports"
PLOT_DIR = PROJECT_ROOT / "artifacts" / "plots"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = PROCESSED_DIR / "cleaned_data.csv"
feature_path = PROCESSED_DIR / "feature_engineered_data.csv"

df = pd.read_csv(cleaned_path)
feature_df = pd.read_csv(feature_path) if feature_path.exists() else df.copy()

print(f"Loaded {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Cleaned data: {len(df):,} rows | {len(df.columns):,} columns")
print(f"Feature data: {len(feature_df):,} rows | {len(feature_df.columns):,} columns")
df.head()

## Dataset Overview

In [ ]:
overview = pd.DataFrame({
    "column_name": df.columns,
    "data_type": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().to_numpy(),
})

overview.to_csv(REPORT_DIR / "dataset_overview.csv", index=False)
overview

## Missing Values

In [ ]:
missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().to_numpy(),
    "missing_percent": (df.isna().mean() * 100).to_numpy(),
    "fill_strategy": "Linear Interpolation + Median",
})

missing_report.to_csv(REPORT_DIR / "missing_value_report.csv", index=False)
missing_report

## Summary Statistics

In [ ]:
signal_columns = [col for col in ["red", "ir", "red_corrected", "ir_corrected"] if col in df.columns]

summary_rows = []
for col in signal_columns:
    mode_value = df[col].mode(dropna=True)
    summary_rows.append({
        "column": col,
        "mean": df[col].mean(),
        "std": df[col].std(),
        "mode": mode_value.iloc[0] if not mode_value.empty else np.nan,
        "min": df[col].min(),
        "max": df[col].max(),
        "median": df[col].median(),
    })

summary_statistics = pd.DataFrame(summary_rows)
summary_statistics.to_csv(REPORT_DIR / "summary_statistics.csv", index=False)
summary_statistics

## Time Continuity

In [ ]:
timestamp_col = "timestamp_ms"
time_diff = df[timestamp_col].diff() if timestamp_col in df.columns else pd.Series(dtype=float)
positive_diffs = time_diff[time_diff > 0]
expected_interval = positive_diffs.mode().iloc[0] if not positive_diffs.mode().empty else np.nan

time_report = pd.DataFrame([
    {"metric": "expected_interval_ms", "value": expected_interval},
    {"metric": "approx_sampling_rate_hz", "value": 1000 / expected_interval if expected_interval else np.nan},
    {"metric": "duplicate_timestamps", "value": df[timestamp_col].duplicated().sum() if timestamp_col in df.columns else np.nan},
    {"metric": "missing_gap_count", "value": (time_diff > expected_interval * 1.5).sum() if expected_interval else np.nan},
    {"metric": "minimum_interval_ms", "value": time_diff.min() if not time_diff.empty else np.nan},
    {"metric": "maximum_interval_ms", "value": time_diff.max() if not time_diff.empty else np.nan},
])

time_report.to_csv(REPORT_DIR / "time_continuity_report.csv", index=False)
time_report

## Outlier Report

In [ ]:
outlier_rows = []
for col in signal_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(outlier_mask.sum()),
        "outlier_percentage": outlier_mask.mean() * 100,
    })

outlier_report = pd.DataFrame(outlier_rows)
outlier_report.to_csv(REPORT_DIR / "outlier_report.csv", index=False)
outlier_report

## Feature Columns

In [ ]:
feature_columns = pd.DataFrame({"feature_columns": feature_df.columns})
feature_columns.to_csv(REPORT_DIR / "feature_engineering_columns.csv", index=False)
feature_columns

## Signal Plots

In [ ]:
x = df[timestamp_col] if timestamp_col in df.columns else df.index

plot_specs = [
    ("red", "Red PPG Signal", "time_series_red.png"),
    ("ir", "IR PPG Signal", "time_series_ir.png"),
    ("red_corrected", "Corrected Red PPG Signal", "time_series_red_corrected.png"),
    ("ir_corrected", "Corrected IR PPG Signal", "time_series_ir_corrected.png"),
]

for column, title, file_name in plot_specs:
    if column not in df.columns:
        continue
    plt.figure(figsize=(14, 4))
    plt.plot(x, df[column], linewidth=0.8)
    plt.title(title)
    plt.xlabel("Timestamp (ms)" if timestamp_col in df.columns else "Sample")
    plt.ylabel(column)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / file_name, dpi=150)
    plt.show()

## Key Observations

- The expected interval is about 20 ms, which corresponds to an approximate sampling rate of 50 Hz.
- Duplicate timestamps and non-20 ms gaps are present, so downstream time-series modeling should account for irregular sampling.
- The corrected channels are centered much closer to zero than the raw red and IR channels, making them useful for pulse-wave morphology analysis.